# D2 — Frozen AraBERT-twitter: all token states → BiLSTM + attention pooling

The encoder is frozen; every token state of the last layer is fed to a BiLSTM, followed by **attention pooling written by hand**:
`αᵢ = softmax(vᵀ tanh(W hᵢ))`, `c = Σ αᵢ hᵢ`. Class-weighted loss, early stopping on val sarcastic F1, threshold tuned on val.

Plan: every preprocessing with seed 42 → best preprocessing re-run with seeds 43 and 44 → attention shown on 5 sarcastic tweets.
Token states are computed once per preprocessing (fp16, kept on CPU). Test stays sealed.

In [ ]:
import os
import sys

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q uv
!uv pip install --system emoji tashaphyne

In [ ]:
# arabert pins an old emoji (1.x, no Arabic names): install it without deps, then force emoji>=2.16.
# If you see a KeyError in emoji.demojize, restart the runtime once and re-run from the top.
%pip install -q --no-deps arabert
%pip install -q -U "emoji>=2.16" pyarabic sentencepiece protobuf

In [ ]:
import copy
import gc
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display
from torch import nn
from transformers import AutoModel, AutoTokenizer

from src.data import get_data
from src.evaluate import ROOT, RESULTS, log_result, metrics, plot_results, save_predictions, show
from src.preprocess import build_pipeline
from src.train import best_threshold, count_params, set_seed

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

In [ ]:
train, val, _, report = get_data()      # test set stays sealed until Part F
y_train, y_val = train["sarcasm"].values, val["sarcasm"].values

counts = train["sarcasm"].value_counts().sort_index()
WEIGHTS = torch.tensor(len(train) / (2 * counts.values), dtype=torch.float).to(DEVICE)
print(f"train: {len(train)} | val: {len(val)} | class weights: {WEIGHTS.tolist()}")

In [ ]:
BASE = ["html", "mentions", "hashtags"]
EMOJI = ["emoji"]
NORM = ["diacritics", "kashida", "letters", "elongation"]
LIGHT = ["light_stem"]
URLS = ["urls"]

COMBOS = {
    # same names / definitions as 02_baseline (so rows are comparable in results.csv)
    "base":                      BASE,
    "emoji":                     BASE + EMOJI,
    "norm":                      BASE + NORM,
    "light_stem":                BASE + LIGHT,
    "emoji + norm":              BASE + EMOJI + NORM,
    "emoji + light_stem":        BASE + EMOJI + LIGHT,
    "norm + light_stem":         BASE + NORM + LIGHT,
    "emoji + norm + light_stem": BASE + EMOJI + NORM + LIGHT,
    "urls kept":                 BASE,
    # added for PLMs: URLs -> <URL>, no stemming, AraBERT's own preprocessor
    "urls replaced":             BASE + URLS,
    "emoji + urls replaced":     BASE + URLS + EMOJI,
    "minimal":                   ["html", "urls", "mentions"],   # what 05_finetune uses
    "arabert":                   None,
}

In [ ]:
def make_prep(steps):
    if steps is None:
        from arabert.preprocess import ArabertPreprocessor
        return ArabertPreprocessor(model_name="aubmindlab/bert-base-arabertv02-twitter").preprocess
    return build_pipeline(steps)


def prepare(frame, steps):
    prep = make_prep(steps)
    texts = [prep(tweet) for tweet in frame["tweet"]]
    return [t if t.strip() else raw for t, raw in zip(texts, frame["tweet"])]


try:
    make_prep(None)("هذا مثال 😂 @user #وسم_تجريبي http://t.co/x")   # fails here, not mid-run, if arabert is incompatible
except Exception as error:
    print("arabert preprocessing skipped:", error)
    COMBOS.pop("arabert")

print(list(COMBOS))
print("base       :", prepare(val.head(4), COMBOS["base"])[3][:100])
print("urls kept == base (BASE has no urls step), so those two rows are expected to match")

In [ ]:
def slug(name):
    return name.replace(" + ", "+").replace(" ", "_")


MODELS = ROOT / "models"
MODELS.mkdir(exist_ok=True)

NAME = "aubmindlab/bert-base-arabertv02-twitter"
MAX_LEN, BATCH = 96, 64

tok = AutoTokenizer.from_pretrained(NAME)
encoder = AutoModel.from_pretrained(NAME).to(DEVICE).eval()
for p in encoder.parameters():
    p.requires_grad = False
HIDDEN = encoder.config.hidden_size

### Model: BiLSTM + attention pooling

In [ ]:
class AttentionPooling(nn.Module):
    """alpha_i = softmax(v^T tanh(W h_i)),  c = sum_i alpha_i h_i  (padding masked)"""

    def __init__(self, dim, hidden=128):
        super().__init__()
        self.W = nn.Linear(dim, hidden)
        self.v = nn.Linear(hidden, 1, bias=False)

    def forward(self, h, mask):
        scores = self.v(torch.tanh(self.W(h))).squeeze(-1)
        alpha = torch.softmax(scores.masked_fill(~mask, -1e9), dim=-1)
        return (alpha.unsqueeze(-1) * h).sum(1), alpha


class BiLSTMAttention(nn.Module):
    def __init__(self, in_dim=HIDDEN, hidden=128, att=128, dropout=0.3):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True, bidirectional=True)
        self.att = AttentionPooling(2 * hidden, att)
        self.out = nn.Linear(2 * hidden, 2)

    def forward(self, x, mask):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.drop(x), mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=x.size(1))
        context, alpha = self.att(self.drop(h), mask)
        return self.out(context), alpha

In [ ]:
@torch.no_grad()
def token_states(texts):
    """Last-layer state of every token: fp16 (N, L, H) + bool mask (N, L), on CPU."""
    L = max(len(ids) for ids in tok(texts, truncation=True, max_length=MAX_LEN)["input_ids"])
    states = torch.zeros(len(texts), L, HIDDEN, dtype=torch.float16)
    mask = torch.zeros(len(texts), L, dtype=torch.bool)
    order = np.argsort([len(t) for t in texts])
    for i in range(0, len(texts), BATCH):
        idx = order[i:i + BATCH]
        enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            hidden = encoder(**enc).last_hidden_state
        width, rows = hidden.size(1), torch.as_tensor(idx)
        states[rows, :width] = hidden.half().cpu()
        mask[rows, :width] = enc["attention_mask"].bool().cpu()
    return states, mask


def to_device(states, mask, idx):
    x, m = states[idx], mask[idx]
    width = int(m.sum(1).max())
    return x[:, :width].to(DEVICE).float(), m[:, :width].to(DEVICE)


@torch.no_grad()
def predict(model, states, mask, batch=256):
    model.eval()
    out = []
    for i in range(0, len(states), batch):
        x, m = to_device(states, mask, torch.arange(i, min(i + batch, len(states))))
        out.append(torch.softmax(model(x, m)[0], dim=-1)[:, 1].cpu())
    return torch.cat(out).numpy()


def fit_head(model, train_data, val_data, epochs=15, patience=3, lr=1e-3, batch=64):
    states, mask = train_data
    y = torch.as_tensor(y_train, dtype=torch.long)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(weight=WEIGHTS)
    best_f1, best_state, wait, start = -1.0, None, 0, time.time()

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(states))
        for i in range(0, len(perm), batch):
            idx = perm[i:i + batch]
            x, m = to_device(states, mask, idx)
            loss = loss_fn(model(x, m)[0], y[idx].to(DEVICE))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        proba = predict(model, *val_data)
        f1 = metrics(y_val, proba, best_threshold(y_val, proba))["f1"]
        if f1 > best_f1:
            best_f1, best_state, wait = f1, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return time.time() - start, epoch + 1

In [ ]:
KEEP = {"f1": -1.0}        # best run so far, used for the attention figure


def run(combos, seeds=(42,)):
    global KEEP
    rows, seen = [], {}

    for name, steps in combos.items():
        train_texts, val_texts = prepare(train, steps), prepare(val, steps)
        h = hash((tuple(train_texts), tuple(val_texts)))
        Xtr = Xva = None

        for seed in seeds:
            if (h, seed) not in seen:           # identical inputs (base / urls kept) are trained once
                if Xtr is None:
                    Xtr, Xva = token_states(train_texts), token_states(val_texts)
                set_seed(seed)
                head = BiLSTMAttention().to(DEVICE)
                seconds, epochs = fit_head(head, Xtr, Xva)

                proba = predict(head, *Xva)
                threshold = best_threshold(y_val, proba)
                seen[(h, seed)] = dict(proba=proba, threshold=threshold, seconds=seconds, epochs=epochs,
                                       scores=metrics(y_val, proba, threshold), params=count_params(head),
                                       state={k: v.cpu() for k, v in head.state_dict().items()})

                if seen[(h, seed)]["scores"]["f1"] > KEEP["f1"]:
                    KEEP = {"f1": seen[(h, seed)]["scores"]["f1"], "name": name, "steps": steps, "seed": seed,
                            "threshold": threshold, "proba": proba, "state": copy.deepcopy(head.state_dict()), "Xva": Xva}

            r = seen[(h, seed)]
            log_result(
                model="arabert_frozen_bilstm_att",
                preprocessing=name,
                seed=seed,
                threshold=r["threshold"],
                train_time=r["seconds"],
                trained_params=r["params"],
                **{f"val_{k}": v for k, v in r["scores"].items()}
            )
            save_predictions(f"arabert_frozen_bilstm_att_{slug(name)}", seed, val["id"], y_val, r["proba"], r["threshold"])

            torch.save({                         # trained head only; the frozen encoder is re-loaded by name
                "state_dict": r["state"], "threshold": float(r["threshold"]),
                "head": dict(in_dim=HIDDEN, hidden=128, att=128, dropout=0.3),
                "encoder": NAME, "max_len": MAX_LEN, "preprocessing": name, "steps": steps, "seed": seed,
                "val_f1": float(r["scores"]["f1"]),
            }, MODELS / f"arabert_frozen_bilstm_att_{slug(name)}_seed{seed}.pt")

            show(f"{name} | seed {seed} | {r['epochs']} ep", r["scores"], r["threshold"], r["seconds"])
            rows.append({"preprocessing": name, "seed": seed, "threshold": r["threshold"], **r["scores"]})

        del Xtr, Xva
        gc.collect()
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)

In [ ]:
table = run(COMBOS)                     # seed 42, every preprocessing
table.set_index("preprocessing")[["f1", "pr_auc", "macro_f1", "precision", "recall"]].round(4)

In [ ]:
best_name = table.loc[table["f1"].idxmax(), "preprocessing"]
print("best preprocessing:", best_name)
more = run({best_name: COMBOS[best_name]}, seeds=(43, 44))      # 3 seeds for the best config

res = pd.read_csv(RESULTS)
runs = res[(res["model"] == "arabert_frozen_bilstm_att") & (res["preprocessing"] == best_name)]
runs[["val_f1", "val_pr_auc", "val_macro_f1"]].agg(["mean", "std"]).round(4)

In [ ]:
plot_results(table.set_index("preprocessing").nlargest(15, "f1"))

### Attention on 5 sarcastic tweets

Best run overall, tweets correctly detected. Darker = more attention. Special tokens removed.

In [ ]:
def attention_html(tokens, alpha):
    top = float(max(alpha)) or 1.0
    spans = [
        f'<span style="background:rgba(255,99,71,{a / top:.2f});padding:2px 4px;margin:2px;'
        f'border-radius:4px;display:inline-block">{t[2:] if t.startswith("##") else t}</span>'
        for t, a in zip(tokens, alpha)
    ]
    return '<div dir="rtl" style="font-size:18px;line-height:2.3">' + "".join(spans) + "</div>"


head = BiLSTMAttention().to(DEVICE)
head.load_state_dict(KEEP["state"])
head.eval()

texts = prepare(val, KEEP["steps"])
states, mask = KEEP["Xva"]
hits = np.where((y_val == 1) & (KEEP["proba"] >= KEEP["threshold"]))[0]
print(f"{KEEP['name']} | seed {KEEP['seed']} | f1={KEEP['f1']:.4f} | {len(hits)} sarcastic tweets detected")

for i in np.random.default_rng(0).choice(hits, 5, replace=False):
    width = int(mask[i].sum())
    with torch.no_grad():
        _, alpha = head(states[i:i + 1, :width].to(DEVICE).float(), mask[i:i + 1, :width].to(DEVICE))
    tokens = tok.convert_ids_to_tokens(tok(texts[i], truncation=True, max_length=MAX_LEN)["input_ids"])
    display(HTML(f"<p>P(sarcasm) = {KEEP['proba'][i]:.2f}</p>"
                 + attention_html(tokens[1:-1], alpha[0, 1:-1].cpu().numpy())))

### Save to disk
Colab wipes `/content` when the session ends: download `models/` (and `results/results.csv`) before leaving.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    !zip -qr /content/models_d2.zip models
    files.download("/content/models_d2.zip")